# 🔥 Train Multi-Task (Cross-Entropy / Softmax — KHÔNG dùng EDL)

**Thực nghiệm M2-CE:** Multi-Task Learning dùng Cross-Entropy thuần (Softmax) cho nhánh phân loại.  
Không có Evidential Deep Learning, không có Dual-Branch viền ban đỏ.

| Thuộc tính | Giá trị |
|---|---|
| Phân loại loss | CrossEntropyLoss (Softmax) |
| Phân đoạn loss | Focal Tversky |
| Nhánh viền | ❌ Tắt (`--no-boundary`) |
| EDL | ❌ Tắt (`--no-edl`) |
| Run name | `run_multitask_pure_ce` |

---

## 🖥️ Bước 1: Kiểm tra GPU

In [ ]:
!nvidia-smi

## 📁 Bước 2: Mount Google Drive & Sync code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/skin-burn-analysis-system'
BRANCH = 'feat/mt-eanet-edl'
REMOTE_URL = 'https://github.com/Triminh15/skin-burn-analysis-system.git'

if not os.path.exists(REPO_DIR):
    print('📥 Cloning repo lần đầu...')
    !git clone --branch {BRANCH} {REMOTE_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    print('🔄 Repo đã có, kéo code mới nhất...')
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout -f {BRANCH}
    !git reset --hard origin/{BRANCH}

!git log --oneline -3

## 📦 Bước 3: Cài thư viện

In [ ]:
%cd /content/skin-burn-analysis-system
!pip install timm segmentation-models-pytorch albumentations --quiet
print('✅ Thư viện đã cài xong!')

## 🗜️ Bước 4: Giải nén Dataset

In [ ]:
%cd /content/skin-burn-analysis-system

import os

# ⚠️ Thay đường dẫn tới file zip trên Drive của bạn
DRIVE_ZIP_PATH = '/content/drive/MyDrive/skin_burn/data_processed.zip'
DATA_DIR = 'data/processed'

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f'✅ Dataset đã có tại {DATA_DIR}, bỏ qua giải nén.')
else:
    print('📂 Đang giải nén dataset...')
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q "{DRIVE_ZIP_PATH}" -d data/
    print('✅ Giải nén xong!')

# Kiểm tra splits
for split in ['train.csv', 'val.csv', 'test.csv']:
    path = f'data/splits/{split}'
    if os.path.exists(path):
        import pandas as pd
        df = pd.read_csv(path)
        print(f'  📋 {split}: {len(df)} mẫu')
    else:
        print(f'  ⚠️ THIẾU: {path}')

## 🔍 Bước 4b: Kiểm tra đường dẫn file dataset (Debug)
Chạy cell này để xác nhận dataset được load đúng trước khi train.

In [ ]:
%cd /content/skin-burn-analysis-system

import pandas as pd
import os

ROOT = '/content/skin-burn-analysis-system'

for split in ['train', 'val', 'test']:
    csv_path = f'data/splits/{split}.csv'
    if not os.path.exists(csv_path):
        print(f'❌ THIẾU FILE: {csv_path}')
        continue
    df = pd.read_csv(csv_path)
    print(f'\n📋 {split}.csv: {len(df)} dòng, cột: {list(df.columns)}')
    if len(df) > 0:
        # Kiểm tra path của 3 mẫu đầu
        for i, row in df.head(3).iterrows():
            img = os.path.join(ROOT, str(row['image_path']))
            msk = os.path.join(ROOT, str(row['mask_path']))
            img_ok = '✅' if os.path.exists(img) else '❌'
            msk_ok = '✅' if os.path.exists(msk) else '❌'
            print(f'  [{i}] img {img_ok} {img}')
            print(f'       msk {msk_ok} {msk}')
        # Đếm số file hợp lệ
        valid = df.apply(
            lambda r: os.path.exists(os.path.join(ROOT, str(r['image_path'])))
                      and os.path.exists(os.path.join(ROOT, str(r['mask_path']))),
            axis=1
        )
        print(f'  ✅ Mẫu hợp lệ: {valid.sum()}/{len(df)}')
        if valid.sum() == 0:
            print(f'  ⚠️  Tất cả file đều không tìm thấy!')
            print(f'  🔎 Path mẫu trong CSV: {row["image_path"]}')
            print(f'  🔎 Các thư mục trong data/: {os.listdir("data/") if os.path.exists("data/") else "data/ không tồn tại"}')

## 🚀 Bước 5: Train Multi-Task CE (Không EDL, Không Dual-Branch)

Lệnh dưới dùng:
- `--no-edl` → Cross-Entropy (Softmax)
- `--no-boundary` → Tắt nhánh dual-branch viền ban đỏ
- `--w-bnd 0.0` → Trọng số viền = 0

In [ ]:
%cd /content/skin-burn-analysis-system

!python src/vision/train_multitask.py \
    --model-type multitask_edl \
    --no-boundary \
    --w-bnd 0.0 \
    --no-edl \
    --cls-loss ce \
    --seg-loss focal_tversky \
    --epochs 35 \
    --batch-size 16 \
    --lr 1e-4 \
    --weight-decay 1e-4 \
    --img-size 256 \
    --patience 10 \
    --w-seg 1.0 \
    --w-cls 0.5 \
    --output-dir outputs/checkpoints/multitask \
    --run-name run_multitask_pure_ce \
    --seed 42

## 📊 Bước 6: Xem kết quả Test

In [ ]:
%cd /content/skin-burn-analysis-system

import json
import os

result_path = 'outputs/checkpoints/multitask/run_multitask_pure_ce/test_results.json'

if not os.path.exists(result_path):
    print(f'❌ Không tìm thấy: {result_path}')
    print('   Hãy kiểm tra lại Bước 5 đã chạy thành công chưa.')
else:
    with open(result_path, 'r', encoding='utf-8') as f:
        r = json.load(f)

    dice      = r.get('dice', 0.0)
    iou       = r.get('iou', 0.0)
    px_acc    = r.get('pixel_accuracy', 0.0)
    cls_acc   = r.get('accuracy', r.get('test_accuracy', 0.0))
    macro_f1  = r.get('macro_f1', r.get('test_macro_f1', 0.0))
    rec3      = r.get('recall_degree_3', r.get('test_recall_degree_3', 0.0))
    loss_val  = r.get('loss_total', r.get('test_loss', None))

    print('=' * 65)
    print('  📋 KẾT QUẢ: M2-CE — Multi-Task Pure (Cross-Entropy / Softmax)')
    print('=' * 65)
    print(f'  🎯 Phân đoạn:')
    print(f'     Test Dice      : {dice:.2%}')
    print(f'     Test IoU       : {iou:.2%}')
    print(f'     Pixel Accuracy : {px_acc:.2%}')
    if loss_val is not None:
        print(f'     Test Loss      : {loss_val:.4f}')
    print(f'  🏷️  Phân loại (CE / Softmax):')
    print(f'     Cls Accuracy   : {cls_acc:.2%}')
    print(f'     Macro F1       : {macro_f1:.2%}')
    print(f'     Recall Độ 3    : {rec3:.2%}')
    print(f'  ℹ️  Params         : ~24.48M | Latency: ~15ms | ROI crop: Không cần')
    print('=' * 65)
    print()
    print('💡 Raw JSON:')
    print(json.dumps(r, indent=2, ensure_ascii=False))

## 💾 Bước 7: Backup checkpoint + kết quả lên Google Drive

In [ ]:
%cd /content/skin-burn-analysis-system

import shutil, os, datetime

# ⚠️ Thay đường dẫn backup trên Drive
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/skin_burn/checkpoints/run_multitask_pure_ce'
SRC_DIR = 'outputs/checkpoints/multitask/run_multitask_pure_ce'

if os.path.exists(SRC_DIR):
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    
    # Copy toàn bộ thư mục run
    for item in os.listdir(SRC_DIR):
        src = os.path.join(SRC_DIR, item)
        dst = os.path.join(DRIVE_BACKUP_DIR, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f'  ✅ Backed up: {item}')
    
    print(f'\n📁 Backup xong tại: {DRIVE_BACKUP_DIR}')
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'🕐 Thời điểm: {ts}')
else:
    print(f'❌ Không tìm thấy thư mục: {SRC_DIR}')
    print('   Hãy chạy lại Bước 5 trước.')

## 🏆 Bước 8: So sánh với mô hình Baseline (Bảng #01)

In [ ]:
%cd /content/skin-burn-analysis-system

import json, os
import pandas as pd

# ---- Baseline từ Bảng #01 (Phân đoạn đơn nhiệm) ----
baseline_rows = [
    {
        'Model': 'U-Net (ResNet-34) + ComboLoss',
        'Kiến trúc': 'Đơn nhiệm (Seg Only)',
        'Dice': '65.19%', 'IoU': '48.36%', 'Pixel Acc': '86.86%',
        'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A',
        'Params': '24.44M', 'Latency': '~15ms', 'ROI Crop': 'Bắt buộc',
    },
    {
        'Model': 'U-Net (ResNet-34) + Focal Tversky',
        'Kiến trúc': 'Đơn nhiệm (Seg Only)',
        'Dice': '65.07%', 'IoU': '48.23%', 'Pixel Acc': '85.17%',
        'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A',
        'Params': '24.44M', 'Latency': '~15ms', 'ROI Crop': 'Bắt buộc',
    },
    {
        'Model': 'U-Net (ConvNeXt-T) + Focal Tversky',
        'Kiến trúc': 'Đơn nhiệm (Seg Only)',
        'Dice': '63.60%', 'IoU': '46.63%', 'Pixel Acc': '83.74%',
        'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A',
        'Params': '31.93M', 'Latency': '~15ms', 'ROI Crop': 'Bắt buộc',
    },
    {
        'Model': 'Attention U-Net++ (ConvNeXt-T)',
        'Kiến trúc': 'Đơn nhiệm (Seg Only)',
        'Dice': '61.57%', 'IoU': '44.48%', 'Pixel Acc': '82.08%',
        'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A',
        'Params': '32.14M', 'Latency': '~15ms', 'ROI Crop': 'Bắt buộc',
    },
]

# ---- Kết quả M2-CE (file này) ----
result_path = 'outputs/checkpoints/multitask/run_multitask_pure_ce/test_results.json'
ce_row = {
    'Model': 'M2-CE: Multi-Task Pure (CrossEntropy) ⭐',
    'Kiến trúc': 'Đa nhiệm CE (Softmax)',
    'Dice': 'N/A', 'IoU': 'N/A', 'Pixel Acc': 'N/A',
    'Cls Acc': 'N/A', 'Macro F1': 'N/A', 'Recall Độ 3': 'N/A',
    'Params': '24.48M', 'Latency': '~15ms', 'ROI Crop': 'Không cần',
}

if os.path.exists(result_path):
    with open(result_path, 'r') as f:
        r = json.load(f)
    ce_row.update({
        'Dice':          f"{r.get('dice', 0):.2%}",
        'IoU':           f"{r.get('iou', 0):.2%}",
        'Pixel Acc':     f"{r.get('pixel_accuracy', 0):.2%}",
        'Cls Acc':       f"{r.get('accuracy', r.get('test_accuracy', 0)):.2%}",
        'Macro F1':      f"{r.get('macro_f1', r.get('test_macro_f1', 0)):.2%}",
        'Recall Độ 3':   f"{r.get('recall_degree_3', r.get('test_recall_degree_3', 0)):.2%}",
    })

df = pd.DataFrame(baseline_rows + [ce_row])

print('=' * 100)
print('🏆 SO SÁNH: M2-CE (Multi-Task CrossEntropy) vs Baseline Bảng #01')
print('=' * 100)
display(df)

os.makedirs('outputs', exist_ok=True)
df.to_csv('outputs/comparison_ce_vs_baseline.csv', index=False, encoding='utf-8')
print('💾 Đã lưu: outputs/comparison_ce_vs_baseline.csv')